# V12 — Shared preprocessing and cohort cache (study design v2, §3.2 and §4.1)
**GPU, about 45–60 min, Internet ON.** Runs the CheXmask U-Net **once** for every image used anywhere in v2 and stores:
- arm A / B / C inputs (224×224, uint8, single channel; training replicates to 3 channels)
- the lung mask in each arm's own frame (needed for LFR and per-arm chance levels)
- radiologist lesion boxes rasterised in each arm's frame (RSNA and VinDr-CXR positives)
- per-image metadata: crop box, scale, mask QC, dHash

Every architecture in V13 then trains on **byte-identical inputs**.
The masking code (`prep_arms`) follows v1 (dilation, feathering, mean-grey fill, 5 % padding) with three changes:
it returns the crop box (so boxes can be mapped), crops a rectangle instead of a short-side-capped square (v1 cut lung tissue
off in 7.8 % of Kermany images, almost all pneumonia), and stores the dilated lung mask for arm A too (one lung definition).

**Inputs:** V11 output (`split_v2.csv`) · Kermany original · RSNA Pneumonia Detection Challenge · NIH Chest X-rays · VinDr-CXR.

In [ ]:
# Decoders for compressed DICOM transfer syntaxes (some VinDr-CXR files); needs Internet = On. Harmless if already present.
import subprocess, sys, os
if os.path.isdir("/kaggle/input"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)

In [ ]:
import os, sys, re, json, glob, random, hashlib, warnings, time
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import cv2
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"
SEG_BACKEND = os.environ.get("SEG_BACKEND", "ianpan")      # "dummy" = ellipse masks, for CPU smoke tests only
INPUT = os.environ.get("V12_INPUT", "/kaggle/input")
OUT = os.environ.get("V12_OUT", "/kaggle/working")
EXPECTED_FP = {"train": "c443e812516ecf30", "val": "db515415832232a1", "test": "e852070e975c6646"}   # canonical V11 Kaggle run (2026-10-17)
K_RSNA, K_NIH, K_VIN = (30, 30, 30) if QUICK_TEST else (1000, 1431, 1000)
VIN_POS_CLASSES = {"Consolidation", "Lung Opacity", "Infiltration"}
VIN_MIN_READERS = 2
NIH_RSNA_DUP_THR = 10                                        # bits of 256; unrelated images differ by ~128
CFG = dict(img_size=224, orig_short_max=512, mask_dilate_frac=0.025, roi_pad_frac=0.05, mask_feather=9,
           fill_mode="mean", raw_mode="resize", crop_mode="rect", mean=[0.4769] * 3, std=[0.2414] * 3)
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

def find_file(name, root=INPUT):
    for r, _, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    return None

def find_dir_with(name, root=INPUT):
    p = find_file(name, root)
    return os.path.dirname(p) if p else None

## 1. Split v2 (from V11) and fingerprint check

In [ ]:
SPLIT_CSV = find_file("split_v2.csv")
assert SPLIT_CSV, "Add the V11 notebook output (split_v2.csv) as an input"
split = pd.read_csv(SPLIT_CSV)
split = split[split.split_v2.fillna("") != ""].reset_index(drop=True)
def fingerprint(paths):
    return hashlib.md5("|".join(sorted(paths)).encode("utf-8")).hexdigest()[:16]
FP = {s: fingerprint(split.loc[split.split_v2 == s, "relpath"]) for s in ("train", "val", "test")}
print("split fingerprints:", FP)
for s, v in EXPECTED_FP.items():
    if v:
        assert FP[s] == v, f"split drift in {s}: {FP[s]} != {v}"
    else:
        print(f"  (no expected fingerprint set for {s}; paste V11 output into EXPECTED_FP to enforce)")

def find_kermany_root():
    env = os.environ.get("KERMANY_ROOT")
    if env: return env
    c = []
    for base in glob.glob(os.path.join(INPUT, "**/"), recursive=True):
        if "__MACOSX" in base: continue
        if all(os.path.isdir(os.path.join(base, s, k)) for s in ("train", "test") for k in ("NORMAL", "PNEUMONIA")):
            c.append(base.rstrip("/\\"))
    assert c, "Kermany tree not found"
    return sorted(c, key=len)[0]
KROOT = find_kermany_root()
if QUICK_TEST:
    split = pd.concat([d.sample(min(len(d), 12), random_state=SEED) for _, d in split.groupby(["split_v2", "label"])],
                      ignore_index=True)
print("Kermany root:", KROOT, "| images to process:", len(split))

## 2. Segmentation model (v1 loader, unchanged)

In [ ]:
import torch, torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if SEG_BACKEND == "ianpan":
    import transformers
    from transformers import AutoModel
    def _load_seg():
        return AutoModel.from_pretrained("ianpan/chest-x-ray-basic", trust_remote_code=True).to(device).eval()
    _orig_finalize = getattr(transformers.modeling_utils.PreTrainedModel, "_finalize_model_loading", None)
    try:
        if _orig_finalize is not None:
            def _safe_finalize(model, *a, **k):
                if not hasattr(model, "all_tied_weights_keys"):
                    model.all_tied_weights_keys = {}
                return _orig_finalize(model, *a, **k)
            transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _safe_finalize
        seg_model = _load_seg()
    except Exception as e:
        if "all_tied_weights_keys" in str(e):
            transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = {}
            seg_model = _load_seg()
        else:
            raise
    finally:
        if _orig_finalize is not None:
            transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _orig_finalize

    @torch.inference_mode()
    def lung_mask(gray_u8, out_hw):
        x = seg_model.preprocess(gray_u8)
        x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).float().to(device)
        logits = F.interpolate(seg_model(x)["mask"], size=out_hw, mode="bilinear", align_corners=False)
        pred = logits.argmax(dim=1)[0].cpu().numpy()
        return ((pred == 1) | (pred == 2)).astype(np.uint8)           # right + left lung; heart (3) excluded
else:
    print("WARNING: dummy segmentation backend (smoke test only)")
    def lung_mask(gray_u8, out_hw):
        H, W = out_hw; yy, xx = np.mgrid[0:H, 0:W]
        m = (((xx - 0.31 * W) / (0.16 * W)) ** 2 + ((yy - 0.48 * H) / (0.32 * H)) ** 2 <= 1) | \
            (((xx - 0.69 * W) / (0.15 * W)) ** 2 + ((yy - 0.50 * H) / (0.31 * H)) ** 2 <= 1)
        return m.astype(np.uint8)

## 3. Image loading and the three arms (v1 code + crop box returned)

In [ ]:
def load_image_any(path, short_max):
    """PNG/JPG/DICOM -> (gray_u8, scale, (W0, H0)); short side reduced to short_max (v1 behaviour)."""
    if path.lower().endswith((".dcm", ".dicom")):
        import pydicom
        dcm = pydicom.dcmread(path)
        arr = dcm.pixel_array.astype(np.float32); arr -= arr.min()
        if arr.max() > 0: arr /= arr.max()
        if getattr(dcm, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
            arr = 1.0 - arr
        pil = Image.fromarray((arr * 255).astype(np.uint8))
    else:
        pil = Image.open(path)
    pil = pil.convert("RGB").convert("L")
    W0, H0 = pil.size; s = 1.0
    if min(W0, H0) > short_max:
        s = short_max / min(W0, H0)
        pil = pil.resize((int(round(W0 * s)), int(round(H0 * s))), Image.BILINEAR)
    return np.asarray(pil).astype(np.uint8), s, (W0, H0)

def prep_arms(gray_u8, lung_u8, cfg):
    """Three arms from one segmentation. Returns {arm: (img224, mask224)}, the B/C crop box (ty0, ty1, tx0, tx1),
    the fallback flag, and the share of (dilated) lung pixels inside the crop.

    Changes from v1 (design §10, 2026-10-17, before any v2 training):
    - B/C crop the padded lung bounding box as a RECTANGLE and resize it to S×S (anisotropic, like arm A).
      v1 squared the box but capped it at the image's short side, which cut lung tissue off in 7.8 % of Kermany images,
      almost all pneumonia (449 vs 7): a class-asymmetric artefact in exactly the arms being compared.
    - The arm-A mask is the same DILATED lung mask as in B/C (v1 stored it undilated), so all arms share one lung definition.
    """
    H, W = lung_u8.shape; short = min(H, W); S = cfg["img_size"]; out = {}
    a_img = cv2.resize(gray_u8, (S, S), interpolation=cv2.INTER_AREA)
    if lung_u8.sum() < 1:                                             # fallback (flagged)
        ones = np.ones((S, S), np.uint8)
        out["raw"] = (a_img, ones); out["roi"] = (a_img, ones); out["lung"] = (a_img, ones)
        return out, (0, H, 0, W), True, 1.0
    dil = max(1, int(round(short * cfg["mask_dilate_frac"])))
    mask_d = cv2.dilate(lung_u8, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dil + 1, 2 * dil + 1)), iterations=1)
    out["raw"] = (a_img, cv2.resize(mask_d, (S, S), interpolation=cv2.INTER_NEAREST).astype(np.uint8))
    soft = mask_d.astype(np.float32); f = cfg["mask_feather"]
    if f and f >= 3:
        f += (f % 2 == 0); soft = cv2.GaussianBlur(soft, (f, f), 0)
    soft = np.clip(soft, 0.0, 1.0)
    fill = cfg["mean"][0] * 255.0 if cfg["fill_mode"] == "mean" else 0.0
    masked = (gray_u8.astype(np.float32) * soft + fill * (1.0 - soft)).clip(0, 255).astype(np.uint8)
    ys, xs = np.where(mask_d > 0)
    pad = int(round(short * cfg["roi_pad_frac"]))
    ty0 = max(0, int(ys.min()) - pad); tx0 = max(0, int(xs.min()) - pad)
    ty1 = min(H, int(ys.max()) + 1 + pad); tx1 = min(W, int(xs.max()) + 1 + pad)
    coverage = float(mask_d[ty0:ty1, tx0:tx1].sum() / max(mask_d.sum(), 1))
    m224 = cv2.resize(mask_d[ty0:ty1, tx0:tx1], (S, S), interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    out["roi"] = (cv2.resize(gray_u8[ty0:ty1, tx0:tx1], (S, S), interpolation=cv2.INTER_AREA), m224)
    out["lung"] = (cv2.resize(masked[ty0:ty1, tx0:tx1], (S, S), interpolation=cv2.INTER_AREA), m224)
    return out, (ty0, ty1, tx0, tx1), False, coverage

def boxes_to_frames(boxes, scale, HW, crop, S=224):
    """boxes in ORIGINAL pixel coords [(x0, y0, x1, y1)] -> {arm: S×S uint8 mask}."""
    H, W = HW; ty0, ty1, tx0, tx1 = crop
    out = {a: np.zeros((S, S), np.uint8) for a in ("raw", "roi", "lung")}
    for (bx0, by0, bx1, by1) in boxes:
        X0, Y0, X1, Y1 = bx0 * scale, by0 * scale, bx1 * scale, by1 * scale            # loaded-image coords
        r = [X0 * S / W, Y0 * S / H, X1 * S / W, Y1 * S / H]                           # arm A: anisotropic resize
        sx, sy = S / (tx1 - tx0), S / (ty1 - ty0)
        c = [(X0 - tx0) * sx, (Y0 - ty0) * sy, (X1 - tx0) * sx, (Y1 - ty0) * sy]       # arms B/C: crop + resize
        for arm, q in (("raw", r), ("roi", c), ("lung", c)):
            a0, b0 = int(np.clip(np.floor(q[0]), 0, S)), int(np.clip(np.floor(q[1]), 0, S))
            a1, b1 = int(np.clip(np.ceil(q[2]), 0, S)), int(np.clip(np.ceil(q[3]), 0, S))
            out[arm][b0:b1, a0:a1] = 1
    return out

def dhash256(gray):
    """16×16 difference hash on a min-max-normalised float image (robust to the DICOM vs PNG intensity stretch)."""
    g = gray.astype(np.float32); g -= g.min(); g /= max(float(g.max()), 1e-6)
    a = cv2.resize(g, (17, 16), interpolation=cv2.INTER_AREA)
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten())

def n_components(m):
    return int(cv2.connectedComponents(m.astype(np.uint8))[0] - 1)

## 4. Cohort item lists
Each item: `(cohort, image_id, path, label, split, boxes_original_px)`; label 1 = pneumonia / opacity.

In [ ]:
items = []
for _, r in split.iterrows():
    items.append(("internal", r.relpath, os.path.join(KROOT, r.relpath), int(r.label == "PNEUMONIA"), r.split_v2, []))

RSNA_BASE = find_dir_with("stage_2_detailed_class_info.csv")
rsna_ids = []
if RSNA_BASE:
    info = pd.read_csv(os.path.join(RSNA_BASE, "stage_2_detailed_class_info.csv")).drop_duplicates("patientId")
    lab = pd.read_csv(os.path.join(RSNA_BASE, "stage_2_train_labels.csv"))
    bx = {pid: [(float(x), float(y), float(x + w), float(y + h)) for x, y, w, h in zip(d.x, d.y, d.width, d.height)]
          for pid, d in lab[lab.Target == 1].groupby("patientId")}
    pos = info[info["class"] == "Lung Opacity"].patientId.tolist(); neg = info[info["class"] == "Normal"].patientId.tolist()
    rng = random.Random(SEED); rng.shuffle(pos); rng.shuffle(neg)          # identical draw to v1 (prefix = v1 sample)
    img_dir = os.path.join(RSNA_BASE, "stage_2_train_images")
    for p in pos[:K_RSNA]: items.append(("rsna", p, os.path.join(img_dir, p + ".dcm"), 1, "ext", bx.get(p, [])))
    for p in neg[:K_RSNA]: items.append(("rsna", p, os.path.join(img_dir, p + ".dcm"), 0, "ext", []))
    rsna_ids = pos[:K_RSNA] + neg[:K_RSNA]
print("RSNA:", len(rsna_ids), "images", "(base:", RSNA_BASE, ")")

NIH_BASE = find_dir_with("Data_Entry_2017.csv")
nih_candidates = {0: [], 1: []}
if NIH_BASE:
    d = pd.read_csv(os.path.join(NIH_BASE, "Data_Entry_2017.csv")); labs = d["Finding Labels"].astype(str)
    index = {}
    for r_, _, files in os.walk(NIH_BASE):
        for f in files:
            if f.lower().endswith(".png"): index[f] = os.path.join(r_, f)
    for lbl, m in ((1, labs.apply(lambda s: "Pneumonia" in s.split("|"))), (0, labs.str.strip() == "No Finding")):
        c = [n for n in d[m]["Image Index"].tolist() if n in index]
        rng = random.Random(SEED); rng.shuffle(c)
        nih_candidates[lbl] = [(n, index[n]) for n in c]
print("NIH candidates:", {k: len(v) for k, v in nih_candidates.items()}, "(base:", NIH_BASE, ")")

def find_vindr():
    csv = None
    for r_, _, files in os.walk(INPUT):
        if "train.csv" in files and "vin" in r_.lower():
            t = pd.read_csv(os.path.join(r_, "train.csv"), nrows=3)
            if {"image_id", "class_name", "rad_id", "x_min"} <= set(t.columns):
                csv = os.path.join(r_, "train.csv"); break
    if not csv: return None, None, None
    dcm = {os.path.splitext(f)[0]: os.path.join(r_, f) for r_, _, fs in os.walk(os.path.dirname(csv))
           for f in fs if f.endswith((".dicom", ".dcm"))}
    if dcm: return csv, dcm, None
    png = {os.path.splitext(f)[0]: os.path.join(r_, f) for r_, _, fs in os.walk(INPUT) if "vin" in r_.lower()
           for f in fs if f.endswith(".png")}
    meta_ = None
    for r_, _, fs in os.walk(INPUT):
        for f in fs:
            if "vin" in r_.lower() and f.endswith(".csv") and "meta" in f.lower():
                meta_ = pd.read_csv(os.path.join(r_, f)); break
    return csv, png, meta_

VIN_CSV, VIN_FILES, VIN_META = find_vindr()
if VIN_CSV:
    v = pd.read_csv(VIN_CSV)
    per = v.groupby("image_id")
    nrad = per.rad_id.nunique()
    allnf = per.class_name.agg(lambda s: (s == "No finding").all())
    tgt = v[v.class_name.isin(VIN_POS_CLASSES)]
    readers = tgt.groupby("image_id").rad_id.nunique()
    pos_ids = [i for i in readers[readers >= VIN_MIN_READERS].index if i in VIN_FILES]
    neg_ids = [i for i in allnf[allnf & (nrad >= 3)].index if i in VIN_FILES]
    boxes_v = {i: list(zip(d.x_min, d.y_min, d.x_max, d.y_max)) for i, d in tgt.groupby("image_id")}
    rng = random.Random(SEED); rng.shuffle(pos_ids); rng.shuffle(neg_ids)
    for i in pos_ids[:K_VIN]: items.append(("vindr", i, VIN_FILES[i], 1, "ext", boxes_v.get(i, [])))
    for i in neg_ids[:K_VIN]: items.append(("vindr", i, VIN_FILES[i], 0, "ext", []))
    print(f"VinDr-CXR: {min(len(pos_ids), K_VIN)} pos / {min(len(neg_ids), K_VIN)} neg (pool {len(pos_ids)} / {len(neg_ids)})")
else:
    print("VinDr-CXR not found: add `vinbigdata-chest-xray-abnormalities-detection` (or a resized-PNG mirror + meta csv)")

def vindr_box_scale(image_id, loaded_wh_original):
    """Boxes in train.csv are in ORIGINAL DICOM pixels. For PNG mirrors, rescale with the meta csv (dim0 = H, dim1 = W)."""
    if VIN_META is None: return 1.0, 1.0
    m = VIN_META[VIN_META.image_id == image_id]
    if not len(m): return 1.0, 1.0
    H0, W0 = float(m.iloc[0]["dim0"]), float(m.iloc[0]["dim1"])
    w, h = loaded_wh_original
    return w / W0, h / H0

## 5. Process everything (segmentation once per image)

In [ ]:
S = CFG["img_size"]
store = {}
meta_rows = []
rsna_hashes = []
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def process(cohort, iid, path, label, spl, boxes):
    gray, s, (W0, H0) = load_image_any(path, CFG["orig_short_max"])
    lung = lung_mask(gray, gray.shape[:2])
    arms, crop, fallback, coverage = prep_arms(gray, lung, CFG)
    if cohort == "vindr" and VIN_META is not None and boxes:
        fx, fy = vindr_box_scale(iid, (W0, H0)); boxes = [(a * fx, b * fy, c * fx, d * fy) for a, b, c, d in boxes]
    bm = boxes_to_frames(boxes, s, gray.shape[:2], crop) if boxes else None
    dh = dhash256(gray)
    st = store.setdefault(cohort, {k: [] for k in ("raw", "roi", "lung", "m_raw", "m_roi", "m_lung", "b_raw", "b_roi", "b_lung")})
    for a in ("raw", "roi", "lung"):
        st[a].append(arms[a][0]); st["m_" + a].append(arms[a][1])
        st["b_" + a].append(bm[a] if bm is not None else np.zeros((S, S), np.uint8))
    row = dict(cohort=cohort, image_id=iid, path=path, label=label, split=spl, W0=W0, H0=H0, scale=s,
               crop_ty0=crop[0], crop_ty1=crop[1], crop_tx0=crop[2], crop_tx1=crop[3], fallback=fallback,
               lung_area_orig=float(lung.mean()), lung_components=n_components(lung),
               n_boxes=len(boxes), dhash=dh.tobytes().hex(), lung_coverage_crop=coverage,
               crop_aspect=(crop[3] - crop[2]) / max(crop[1] - crop[0], 1))
    for a in ("raw", "roi", "lung"):
        m = arms[a][1].astype(bool); row[f"lung_frac_{a}"] = float(m.mean())
        if bm is not None:
            b = bm[a].astype(bool); row[f"box_frac_{a}"] = float(b.mean())
            row[f"box_in_lung_frac_{a}"] = float((b & m).sum() / max(m.sum(), 1))
            row[f"box_inside_lung_share_{a}"] = float((b & m).sum() / max(b.sum(), 1))
    meta_rows.append(row)
    return dh

t0 = time.time(); fails = []
for k, it in enumerate(items):
    try:
        dh = process(*it)
        if it[0] == "rsna": rsna_hashes.append(dh)
    except Exception as e:
        fails.append((it[0], it[1], repr(e)[:200]))
    if k % 500 == 0: print(f"  {k}/{len(items)}  {time.time() - t0:.0f}s")

# NIH: sampled after RSNA so RSNA duplicates can be skipped while filling the quota
RH = np.stack(rsna_hashes) if rsna_hashes else np.zeros((0, 32), np.uint8)
nih_dupes = []
for lbl in (1, 0):
    taken = 0
    for name, path in nih_candidates[lbl]:
        if taken >= K_NIH: break
        try:
            gray, _, _ = load_image_any(path, CFG["orig_short_max"])
            dh = dhash256(gray)
            if len(RH) and int(POP[np.bitwise_xor(RH, dh[None])].sum(-1).min()) <= NIH_RSNA_DUP_THR:
                nih_dupes.append(name); continue
            process("nih", name, path, lbl, "ext", []); taken += 1
        except Exception as e:
            fails.append(("nih", name, repr(e)[:200]))
print(f"done in {time.time() - t0:.0f}s | failures: {len(fails)} | NIH images skipped as RSNA duplicates: {len(nih_dupes)}")

## 6. Save the cache, metadata, chance levels, and QC

In [ ]:
meta = pd.DataFrame(meta_rows)
meta["mask_suspect"] = (meta.lung_area_orig < 0.15) | (meta.lung_area_orig > 0.45)
meta["mask_tiny"] = meta.lung_area_orig < 0.08
meta["row"] = -1
for cohort, st in store.items():
    sel = meta.cohort == cohort
    meta.loc[sel, "row"] = np.arange(int(sel.sum()))
    np.savez_compressed(os.path.join(OUT, f"cache_{cohort}.npz"),
                        **{k: np.stack(v).astype(np.uint8) for k, v in st.items()})
meta["row"] = meta["row"].astype(int)
cov_min = float(meta.lung_coverage_crop.min())
print(f"lung coverage of the B/C crop: min {cov_min:.4f} (must be ≥ 0.99) | crop aspect W/H median by class:",
      meta.groupby(["cohort", "label"]).crop_aspect.median().round(3).to_dict())
assert cov_min >= 0.99, "B/C crop loses lung tissue"
meta.to_csv(os.path.join(OUT, "manifest_v2.csv"), index=False)
pd.DataFrame(fails, columns=["cohort", "image_id", "error"]).to_csv(os.path.join(OUT, "v12_failures.csv"), index=False)
pd.Series(nih_dupes, name="nih_image_index", dtype=str).to_csv(os.path.join(OUT, "v12_nih_rsna_duplicates.csv"), index=False)

ch = []
for cohort in meta.cohort.unique():
    for arm in ("raw", "roi", "lung"):
        m = meta[meta.cohort == cohort]; pos = m[(m.label == 1) & (m.n_boxes > 0)]
        ch.append(dict(cohort=cohort, arm=arm, n=len(m), lfr_chance=m[f"lung_frac_{arm}"].mean(),
                       n_box_pos=len(pos),
                       lefr_chance=pos[f"box_frac_{arm}"].mean() if len(pos) else np.nan,
                       lefr_in_lung_chance=pos[f"box_in_lung_frac_{arm}"].mean() if len(pos) else np.nan,
                       box_inside_lung_share=pos[f"box_inside_lung_share_{arm}"].mean() if len(pos) else np.nan))
chance = pd.DataFrame(ch); chance.to_csv(os.path.join(OUT, "chance_levels_v2.csv"), index=False)
print(chance.round(3).to_string(index=False))

qc = meta.groupby("cohort").agg(n=("label", "size"), pos=("label", "sum"), fallback=("fallback", "sum"),
                                 suspect=("mask_suspect", "mean"), tiny=("mask_tiny", "mean"),
                                 lung_area=("lung_area_orig", "mean"),
                                 two_components=("lung_components", lambda s: (s == 2).mean()))
qc.to_csv(os.path.join(OUT, "v12_mask_qc.csv")); print(qc.round(3))
summary = dict(seed=SEED, cfg=CFG, quick_test=QUICK_TEST, seg_backend=SEG_BACKEND, split_fingerprints=FP,
               counts={f"{c}|label{l}": int(n) for (c, l), n in meta.groupby(["cohort", "label"]).size().items()},
               failures=len(fails), nih_rsna_duplicates_skipped=len(nih_dupes),
               vindr_rule=dict(classes=sorted(VIN_POS_CLASSES), min_readers=VIN_MIN_READERS, negative="all 3 readers: No finding"))
with open(os.path.join(OUT, "v12_summary.json"), "w") as fh:
    json.dump(summary, fh, indent=2, default=str)

# QC figure: 3 positives per cohort × 3 arms, lung contour (teal) + lesion box (yellow dashed)
cohorts = [c for c in ("internal", "rsna", "nih", "vindr") if c in store]
fig, axs = plt.subplots(len(cohorts), 9, figsize=(7.16, 0.9 * len(cohorts) + 0.3), squeeze=False)
for i, c in enumerate(cohorts):
    m = meta[(meta.cohort == c) & (meta.label == 1)]
    m = m[m.n_boxes > 0] if (m.n_boxes > 0).any() else m
    rows = m.sample(min(3, len(m)), random_state=SEED).row.tolist() if len(m) else []
    for j in range(3):
        for k_, a in enumerate(("raw", "roi", "lung")):
            ax = axs[i, j * 3 + k_]; ax.set_axis_off()
            if j < len(rows):
                rr = rows[j]
                ax.imshow(store[c][a][rr], cmap="gray", vmin=0, vmax=255)
                ax.contour(store[c]["m_" + a][rr], levels=[0.5], colors="#0E8FA0", linewidths=0.5)
                if store[c]["b_" + a][rr].any():
                    ax.contour(store[c]["b_" + a][rr], levels=[0.5], colors="#F2C14E", linewidths=0.5, linestyles="--")
            if i == 0: ax.set_title({"raw": "A", "roi": "B", "lung": "C"}[a], fontsize=6)
    axs[i, 0].text(-0.15, 0.5, c, transform=axs[i, 0].transAxes, rotation=90, ha="right", va="center", fontsize=6)
fig.savefig(os.path.join(OUT, "fig_v12_qc.png"), dpi=300, bbox_inches="tight"); plt.close(fig)
print("V12 complete. Outputs:", sorted(os.listdir(OUT)))